# FraudTrace AI - 03: Pairwise Feature Engineering for Evidence Correlation

> **DISCLAIMER: SYNTHETIC DATA ONLY**
> All data used in this notebook is synthetically generated for research, testing, and demonstration purposes. No real police records, personal identification, or banking transactions are included. This system does not make legal guilt determinations; it reconstructs and correlates evidentiary relationships.

### Purpose
Construct the 14-dimensional pairwise forensic feature vector comparing evidence pairs.

| Attribute | Description |
| :--- | :--- |
| **Input** | Pairs of cleaned evidence items |
| **Output** | 14-dimensional numerical feature matrix |
| **Target Environment** | Google Colab (CPU / GPU compatible) & Local |
| **Package** | `ml.src` |


In [ ]:
# ====================================================================
# GOOGLE COLAB ENVIRONMENT SETUP
# ====================================================================
import os
import sys

# Clone repository if running inside Google Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('FraudTrace.AI'):
        !git clone https://github.com/Cyberdude441/FraudTrace.AI.git
    %cd FraudTrace.AI
    !pip install -r ml/requirements-colab.txt

# Ensure repository root is on sys.path for local and Colab execution
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"[OK] Environment initialized. Working directory: {os.getcwd()}")


## 1. Feature Representation Overview
To classify relationships between two evidence items, we extract 14 explainable forensic signals:
1. `amount_exact_match` (Binary)
2. `amount_delta_ratio` (Relative percentage delta)
3. `amount_abs_diff` (Absolute monetary delta)
4. `time_drift_seconds` (Continuous temporal gap)
5. `time_within_15min` (Binary temporal window)
6. `time_within_1hour` (Binary temporal window)
7. `time_within_24hours` (Binary temporal window)
8. `same_source_type` (Binary check)
9. `independent_sources` (Binary check)
10. `entity_overlap_count` (Shared resolved entities)
11. `entity_overlap_ratio` (Jaccard entity similarity)
12. `text_levenshtein_sim` (Normalized string edit distance)
13. `text_token_jaccard` (Token word overlap)
14. `has_missing_critical_data` (Indicator of missing required fields)

In [ ]:
from ml.src.preprocessing.feature_builder import PairwiseFeatureBuilder

builder = PairwiseFeatureBuilder()
feature_names = builder.get_feature_names()
print(f"Number of pairwise features: {len(feature_names)}")
for i, fn in enumerate(feature_names, 1):
    print(f" {i:>2}. {fn}")


## 2. Extract Features on Sample Evidence Pairs
We test feature extraction on two contrasting pairs:
- Pair A: Mismatched amounts (₹50k vs ₹48k)
- Pair B: Corroborating amounts (₹50k vs ₹50k, 2 minutes apart)

In [ ]:
import pandas as pd

pair_conflict_1 = {
    "type": "FIR", "source": "Complainant Statement",
    "amount": 50000.0, "timestamp": "2024-03-15T10:30:00Z",
    "entities": [{"type": "phone", "value": "9876543210"}],
    "text": "Transferred 50000 to fraudster phone"
}
pair_conflict_2 = {
    "type": "BANK_STATEMENT", "source": "HDFC Core",
    "amount": 48000.0, "timestamp": "2024-03-15T10:32:00Z",
    "entities": [{"type": "phone", "value": "9876543210"}],
    "text": "Debit transfer 48000 to UPI handle"
}

vec_conflict = builder.build_feature_vector(pair_conflict_1, pair_conflict_2)

pair_corrob_1 = pair_conflict_1
pair_corrob_2 = {
    "type": "SMS_GATEWAY", "source": "Telecom Gateway",
    "amount": 50000.0, "timestamp": "2024-03-15T10:31:00Z",
    "entities": [{"type": "phone", "value": "9876543210"}],
    "text": "Alert: 50000 debited from account"
}
vec_corrob = builder.build_feature_vector(pair_corrob_1, pair_corrob_2)

df_features = pd.DataFrame([
    dict(zip(feature_names, vec_conflict)),
    dict(zip(feature_names, vec_corrob))
], index=["Conflicting Pair (50k vs 48k)", "Corroborating Pair (50k vs 50k)"])

df_features.T


## 3. Key Observations
- Notice how `amount_exact_match` switches from 0.0 to 1.0, and `amount_delta_ratio` collapses to 0.0.
- Notice how `time_drift_seconds` is 120s vs 60s, both cleanly within `time_within_15min`.
- These 14 features provide sufficient signal for the ML contradiction classifier.